In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from collections import defaultdict
from dataloader import HDF5Dataset
from utils.misc import setup_seed
from MAE_model_downstream import PedSleepMAE
import re

# ========== CONFIG ==========
seed = 42
setup_seed(seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# anonymized directories
directory_path = "/data/hdf5_files"
save_dir = "./output_embeddings"
os.makedirs(save_dir, exist_ok=True)

search_labels = ['apnea_label', 'sleep_label', 'desat_label', 'eeg_label', 'hypop_label']
patch_size = 8
mask_ratio = 15
emb_dim = 64
num_head = 4
num_layer = 3
batch_size = 50

pair_range = slice(0, 3)
target_pairs = None  # use pair_range

# ========== Load Model ==========
model = PedSleepMAE(batch_size=batch_size, patch_size=patch_size, mask_ratio=mask_ratio,
                    emb_dim=emb_dim, num_head=num_head, num_layer=num_layer).to(device)
checkpoint = torch.load(f"./checkpoints/signalmask{mask_ratio}_patch{patch_size}.pt", weights_only=True)
model.load_state_dict(checkpoint['state_dict'])
model.eval()

# ========== Group files by (subject_id, session_id) ==========
def extract_sample_id(filename):
    match = re.search(r"_sample_(\d+)\.hdf5$", filename)
    return int(match.group(1)) if match else float("inf")

grouped_files = defaultdict(list)
for fname in os.listdir(directory_path):
    if not fname.endswith(".hdf5"):
        continue
    parts = fname.split("_")
    if len(parts) < 4:
        continue
    key = (parts[0], parts[1])
    grouped_files[key].append(os.path.join(directory_path, fname))

for key in grouped_files:
    grouped_files[key] = sorted(grouped_files[key], key=extract_sample_id)

all_keys = sorted(grouped_files.keys(), key=lambda x: (int(x[0]), int(x[1])))
if target_pairs:
    selected_keys = [key for key in all_keys if key in target_pairs]
else:
    selected_keys = all_keys[pair_range]

print(f"Total valid pairs found: {len(all_keys)}")
print(f"Selected {len(selected_keys)} pairs to extract:\n{selected_keys}")

pool = nn.AdaptiveMaxPool1d(1)

# ========== Extraction ==========
for idx, (subject_id, session_id) in enumerate(selected_keys):
    pid = f"{subject_id}_{session_id}"
    print(f"\nProcessing Pair {idx + 1}/{len(selected_keys)}: {pid}")
    session_files = grouped_files[(subject_id, session_id)]
    dataset = HDF5Dataset(session_files, search_labels)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_embeddings = []
    all_labels = {label: [] for label in search_labels}

    with torch.no_grad():
        for signals, label_dict, ids in loader:
            signals = signals.squeeze().float().to(device)
            encoded, _ = model.encoder(signals)
            encoded = encoded[:, :, 1:, :].reshape(signals.size(0), -1, emb_dim)
            pooled = pool(encoded).squeeze(dim=2).cpu().numpy()
            all_embeddings.append(pooled)
            for label_name in search_labels:
                all_labels[label_name].append(label_dict[label_name].cpu().numpy())

    emb_array = np.vstack(all_embeddings)
    np.save(os.path.join(save_dir, f"{pid}_embeddings.npy"), emb_array)

    for label_name in search_labels:
        label_array = np.concatenate(all_labels[label_name], axis=0)
        np.save(os.path.join(save_dir, f"{pid}_{label_name}.npy"), label_array)

    print(f"  Saved: {pid}_embeddings.npy and label files.")

In [ ]:
import os
import re
from collections import defaultdict

directory_path = "./hdf5_data"
preview_n = 30

def extract_sample_id(filename):
    match = re.search(r"_sample_(\d+)\.hdf5$", filename)
    return int(match.group(1)) if match else float("inf")

grouped_files = defaultdict(list)
for fname in os.listdir(directory_path):
    if not fname.endswith(".hdf5"):
        continue
    parts = fname.split("_")
    if len(parts) < 4:
        continue
    key = (parts[0], parts[1])
    grouped_files[key].append(fname)

for key in grouped_files:
    grouped_files[key] = sorted(grouped_files[key], key=extract_sample_id)

all_keys = sorted(grouped_files.keys(), key=lambda x: (int(x[0]), int(x[1])))

print(f"\nPreviewing first {preview_n} pairs:")
for i, (sub_id, sess_id) in enumerate(all_keys[:preview_n]):
    print(f"{i+1:2d}. Subject: {sub_id}, Session: {sess_id}")

print(f"\nTotal pairs found: {len(all_keys)}")

In [ ]:
# Extracting phate features
import os
import re
from collections import defaultdict

directory_path = "./hdf5_data"
preview_n = 30

def extract_sample_id(filename):
    match = re.search(r"_sample_(\d+)\.hdf5$", filename)
    return int(match.group(1)) if match else float("inf")

grouped_files = defaultdict(list)
for fname in os.listdir(directory_path):
    if fname.endswith(".hdf5"):
        parts = fname.split("_")
        if len(parts) >= 4:
            grouped_files[(parts[0], parts[1])].append(fname)

for key in grouped_files:
    grouped_files[key] = sorted(grouped_files[key], key=extract_sample_id)

all_keys = sorted(grouped_files.keys(), key=lambda x: (int(x[0]), int(x[1])))

print(f"\nPreviewing first {preview_n} pairs:")
for i, (sub_id, sess_id) in enumerate(all_keys[:preview_n]):
    print(f"{i+1:2d}. Subject: {sub_id}, Session: {sess_id}")

print(f"\nTotal pairs found: {len(all_keys)}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from matplotlib.cm import get_cmap

pair_id = "sub01_sess01"  
embedding_dir = "./output_embeddings"

embeddings = np.load(os.path.join(embedding_dir, f"{pair_id}_embeddings.npy"))
time_indices = np.arange(len(embeddings))

tsne_path = os.path.join(embedding_dir, f"{pair_id}_tsne_traj.npy")
phate_path = os.path.join(embedding_dir, f"{pair_id}_phate_traj.npy")
umap_path = os.path.join(embedding_dir, f"{pair_id}_umap_traj.npy")

tsne = np.load(tsne_path) if os.path.exists(tsne_path) else None
phate = np.load(phate_path) if os.path.exists(phate_path) else None
umap = np.load(umap_path) if os.path.exists(umap_path) else None

cmap = get_cmap("turbo")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if tsne is not None:
    sc1 = axes[0].scatter(tsne[:, 0], tsne[:, 1], c=time_indices, cmap=cmap, s=10)
    axes[0].set_title("t-SNE Trajectory")
    fig.colorbar(sc1, ax=axes[0], label="Time Index")

if phate is not None:
    sc2 = axes[1].scatter(phate[:, 0], phate[:, 1], c=time_indices, cmap=cmap, s=10)
    axes[1].set_title("PHATE Trajectory")
    fig.colorbar(sc2, ax=axes[1], label="Time Index")

if umap is not None:
    sc3 = axes[2].scatter(umap[:, 0], umap[:, 1], c=time_indices, cmap=cmap, s=10)
    axes[2].set_title("UMAP Trajectory")
    fig.colorbar(sc3, ax=axes[2], label="Time Index")

plt.suptitle("Trajectory Visualization", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
from glob import glob

feature_dir = "./output_embeddings"
feature_files = sorted(glob(os.path.join(feature_dir, "*_time_feature.npy")))

features, ids = [], []
for file in feature_files:
    sid = os.path.basename(file).replace("_time_feature.npy", "")
    try:
        feat = np.load(file)
        if feat.shape[0] == 10:
            features.append(feat)
            ids.append(sid)
        else:
            print(f"Skipping {sid}: invalid length {feat.shape}")
    except Exception as e:
        print(f"Error loading {file}: {e}")

columns = [
    "total_distance", "avg_distance", "var_distance", "max_distance",
    "num_clusters", "avg_cluster_duration",
    "entropy_dir_change", "mean_angle",
    "straightness", "tortuosity"
]

df = pd.DataFrame(features, columns=columns, index=ids)
df.index.name = "id"
print(f"Loaded {len(df)} feature files")

In [ ]:
import matplotlib.pyplot as plt

df.hist(bins=30, figsize=(15, 10), layout=(3, 4))
plt.suptitle("Distribution of PHATE Time Features")
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between PHATE Trajectory Features")
plt.show()

In [ ]:
## Try Multiple DBSCAN Settings
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN

phate_file = "./output_embeddings/sub01_sess01_phate_traj.npy"
phate_array = np.load(phate_file)

def try_dbscan(arr, eps_list, min_samples_list):
    print(f"Trajectory shape: {arr.shape}")
    for eps in eps_list:
        for min_samples in min_samples_list:
            clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(arr)
            labels = clustering.labels_
            n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
            n_noise = np.sum(labels == -1)
            print(f"[eps={eps:.1f}, min_samples={min_samples}] --> clusters: {n_clusters}, noise points: {n_noise}")
            plot_clusters(arr, labels, eps, min_samples)

def plot_clusters(arr, labels, eps, min_samples):
    plt.figure(figsize=(6, 5))
    plt.scatter(arr[:, 0], arr[:, 1], c=labels, cmap='tab10', s=15)
    plt.title(f"DBSCAN Clusters (eps={eps}, min_samples={min_samples})")
    plt.xlabel("PHATE 1")
    plt.ylabel("PHATE 2")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

eps_values = [0.5, 1.0, 1.5, 2.0, 3.0]
min_samples_values = [3, 5, 10]

try_dbscan(phate_array, eps_values, min_samples_values)

In [ ]:
## Run code to get time features
import os
import numpy as np
from glob import glob
from scipy.stats import entropy
from numpy.linalg import norm
import ruptures as rpt
import math

feature_dir = "./output_embeddings"

def compute_trajectory_features(arr, max_bkps=5):
    distances = norm(np.diff(arr, axis=0), axis=1)
    total_distance = np.sum(distances)
    avg_distance = np.mean(distances)
    var_distance = np.var(distances)
    max_distance = np.max(distances)

    directions = np.diff(arr, axis=0)
    angles = []
    for i in range(1, len(directions)):
        v1, v2 = directions[i - 1], directions[i]
        cosine_angle = np.dot(v1, v2) / (norm(v1) * norm(v2) + 1e-8)
        angle = math.acos(np.clip(cosine_angle, -1, 1))
        angles.append(angle)
    hist, _ = np.histogram(angles, bins=20, range=(0, np.pi), density=True)
    entropy_dir_change = entropy(hist + 1e-8)
    mean_angle = np.mean(angles)

    model = rpt.KernelCPD(kernel="linear").fit(arr)
    bkps = model.predict(pen=0.1)
    num_segments = len(bkps)
    segment_lengths = np.diff([0] + bkps)
    avg_segment_duration = np.mean(segment_lengths) if len(segment_lengths) > 0 else 0

    displacement = norm(arr[-1] - arr[0])
    straightness = displacement / (total_distance + 1e-8)
    tortuosity = total_distance / (displacement + 1e-8)

    return np.array([
        total_distance, avg_distance, var_distance, max_distance,
        num_segments, avg_segment_duration,
        entropy_dir_change, mean_angle,
        straightness, tortuosity
    ])

traj_files = sorted(glob(os.path.join(feature_dir, "*_phate_traj.npy")))

for file in traj_files:
    sid = os.path.basename(file).replace("_phate_traj.npy", "")
    print(f"Processing: {sid}")
    arr = np.load(file)
    if len(arr) < 3:
        print(f"  Skipped {sid} (too few points)")
        continue
    feats = compute_trajectory_features(arr, max_bkps=5)
    outfile = os.path.join(feature_dir, f"{sid}_time_feature.npy")
    np.save(outfile, feats)
    print(f"  Saved: {outfile}")

In [ ]:
import matplotlib.pyplot as plt

df.hist(bins=30, figsize=(15, 10), layout=(3, 4))
plt.suptitle("Distribution of PHATE Time Features")
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
import seaborn as sns

plt.figure(figsize=(7, 5))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between PHATE Trajectory Features")
plt.show()


In [ ]:
import os
import numpy as np
from glob import glob
from sklearn.preprocessing import StandardScaler

feature_dir = "./output_embeddings"
feature_files = sorted(glob(os.path.join(feature_dir, "*_time_feature.npy")))

features, ids = [], []
for file in feature_files:
    sid = os.path.basename(file).replace("_time_feature.npy", "")
    feat = np.load(file)
    if feat.shape[0] == 6:
        features.append(feat)
        ids.append(sid)
    else:
        print(f"Skipped {sid} due to invalid shape {feat.shape}")

features = np.array(features)

scaler = StandardScaler()
normalized = scaler.fit_transform(features)

for i, sid in enumerate(ids):
    out_path = os.path.join(feature_dir, f"{sid}_time_feature_normalized.npy")
    np.save(out_path, normalized[i])
    print(f"Saved: {out_path}")

print("Done.")

In [ ]:
## add new point based feature
import os
import numpy as np
from glob import glob
from scipy.stats import entropy
from numpy.linalg import norm
import ruptures as rpt
import math

# CONFIG
feature_dir = "./output_embeddings"
pair_range = slice(0, 1000)

# FUNCTIONS
def compute_trajectory_features(arr, max_bkps=5):
    distances = norm(np.diff(arr, axis=0), axis=1)
    avg_distance = np.mean(distances)
    max_distance = np.max(distances)

    directions = np.diff(arr, axis=0)
    angles = []
    for i in range(1, len(directions)):
        v1, v2 = directions[i - 1], directions[i]
        cosine = np.dot(v1, v2) / (norm(v1) * norm(v2) + 1e-8)
        angles.append(math.acos(np.clip(cosine, -1, 1)))
    hist, _ = np.histogram(angles, bins=20, range=(0, np.pi), density=True)
    entropy_dir_change = entropy(hist + 1e-8)
    mean_angle = np.mean(angles)

    model = rpt.KernelCPD(kernel="linear").fit(arr)
    bkps = model.predict(pen=0.1)
    num_segments = len(bkps)

    tortuosity = np.sum(distances) / (norm(arr[-1] - arr[0]) + 1e-8)

    delta_distances = np.insert(distances, 0, 0.0)
    cumulative_distances = np.cumsum(delta_distances)
    dist_to_start = norm(arr - arr[0], axis=1)

    angle_changes = np.zeros(len(arr))
    curvatures = np.zeros(len(arr))
    for i in range(1, len(arr) - 1):
        a = arr[i] - arr[i - 1]
        b = arr[i + 1] - arr[i]
        cosine = np.dot(a, b) / (norm(a) * norm(b) + 1e-8)
        angle_changes[i] = math.acos(np.clip(cosine, -1, 1))
        curvatures[i] = (norm(a) + norm(b)) / (norm(arr[i + 1] - arr[i - 1]) + 1e-8)

    segment_ids = np.zeros(len(arr), dtype=int)
    for i, bkp in enumerate(bkps):
        segment_ids[:bkp] = i

    session_features = np.array([
        avg_distance, max_distance,
        num_segments, entropy_dir_change, mean_angle,
        tortuosity
    ])

    point_features = np.stack([
        delta_distances,
        cumulative_distances,
        angle_changes,
        curvatures,
        dist_to_start,
        segment_ids
    ], axis=1)

    return session_features, point_features

# MAIN
traj_files = sorted(glob(os.path.join(feature_dir, "*_phate_traj.npy")))
pair_ids = [os.path.basename(f).replace("_phate_traj.npy", "") for f in traj_files]
pair_ids = sorted(pair_ids, key=lambda x: (int(x.split("_")[0]), int(x.split("_")[1])))
selected_ids = pair_ids[pair_range]

summary_features = []
for idx, sid in enumerate(selected_ids):
    file = os.path.join(feature_dir, f"{sid}_phate_traj.npy")
    arr = np.load(file)
    if len(arr) < 3:
        print(f"[{idx}] Skipped {sid} (too few points)")
        continue

    print(f"[{idx}] Processing {sid}")
    session_feats, point_feats = compute_trajectory_features(arr)
    np.save(os.path.join(feature_dir, f"{sid}_time_feature.npy"), session_feats)
    np.save(os.path.join(feature_dir, f"{sid}_point_features.npy"), point_feats)
    summary_features.append(session_feats)

print(f"Finished processing {len(summary_features)} sessions.")


In [ ]:
# Correlation and distribution plots
df = pd.DataFrame(summary_features, columns=[
    "avg_distance", "max_distance",
    "num_segments", "entropy_dir_change", "mean_angle", "tortuosity"
])

plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", center=0)
plt.title("Correlation Between Updated PHATE Trajectory Features")
plt.tight_layout()
plt.show()

df.hist(bins=20, figsize=(12, 10), layout=(3, 3))
plt.suptitle("Distribution of Updated PHATE Time Features")
plt.tight_layout()
plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob

feature_dir = "./output_embeddings"

point_files = sorted(glob(os.path.join(feature_dir, "*_point_features.npy")))
all_point_features = []

for file in point_files:
    arr = np.load(file)
    if arr.ndim == 2 and arr.shape[1] == 6:
        all_point_features.append(arr)

all_data = np.vstack(all_point_features)
columns = [
    "delta_distance",
    "cumulative_distance",
    "angle_change",
    "local_curvature",
    "dist_to_start",
    "segment_id"
]
df = pd.DataFrame(all_data, columns=columns)

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(columns):
    sns.histplot(df[col], bins=50, kde=False, ax=axes[i])
    axes[i].set_title(f"Distribution of {col}")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

corr = df.corr()
plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Between Point Features")
plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob

feature_dir = "./output_embeddings"

time_files = sorted(glob(os.path.join(feature_dir, "*_time_feature.npy")))
all_time_features = []

for file in time_files:
    arr = np.load(file)
    if arr.ndim == 1 and arr.shape[0] == 6:
        all_time_features.append(arr)

all_time_data = np.vstack(all_time_features)
columns = [
    "avg_distance",
    "max_distance",
    "num_segments",
    "entropy_dir_change",
    "mean_angle",
    "tortuosity"
]
df_time = pd.DataFrame(all_time_data, columns=columns)

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(columns):
    sns.histplot(df_time[col], bins=50, kde=False, ax=axes[i], edgecolor=None)
    axes[i].set_title(f"Distribution of {col}")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

corr_time = df_time.corr()
plt.figure(figsize=(5, 4))
sns.heatmap(corr_time, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Between Features")
plt.show()

In [ ]:
import os
import numpy as np
from glob import glob
from sklearn.preprocessing import StandardScaler

feature_dir = "./output_embeddings"
feature_files = sorted(glob(os.path.join(feature_dir, "*_point_features.npy")))

all_features, ids, lengths = [], [], []

for file in feature_files:
    sid = os.path.basename(file).replace("_point_features.npy", "")
    feat = np.load(file)  # shape: (T, 6)
    if feat.ndim == 2 and feat.shape[1] == 6:
        all_features.append(feat)
        ids.append(sid)
        lengths.append(feat.shape[0])
    else:
        print(f"Skipped {sid} due to invalid shape {feat.shape}")

stacked = np.vstack(all_features)

scaler = StandardScaler()
normalized_stacked = scaler.fit_transform(stacked)

idx = 0
for sid, length in zip(ids, lengths):
    norm_feat = normalized_stacked[idx:idx + length]
    idx += length
    out_path = os.path.join(feature_dir, f"{sid}_point_features_normalized.npy")
    np.save(out_path, norm_feat)
    print(f"Saved: {out_path}")

print("Done.")